# Assignment: API Data Visualization

**Student Name:** Peiru Liang

**Student ID:** EK6339586

**Date:** 28/11/2025

**Chosen API:** REE

---

## Part 1: API Selection and Justification

**Write 1-2 paragraphs explaining:**
- Which API you chose and why
- What type of data you plan to analyze
- Why this data is interesting or relevant

[Your answer here]

---

## Part 2: Setup and Imports

In [7]:
# Import required libraries
import requests
import json
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import matplotlib
from pprint import pprint

# Optional: pandas for data manipulation
# import pandas as pd

# Set matplotlib style
matplotlib.rc('xtick', labelsize=11)
matplotlib.rc('ytick', labelsize=11)
plt.style.use('ggplot')

print("✓ Libraries imported successfully!")

✓ Libraries imported successfully!


### API Configuration

⚠️ **Important:** Replace with your actual API credentials if required

In [9]:
import requests
import pandas as pd

def fetch_historical_weather(lat, lon, start_date, end_date,
                             hourly_vars="temperature_2m,precipitation",
                             timezone="Europe/Madrid"):
    """
    使用 Open-Meteo 历史天气 API (archive-api.open-meteo.com) 获取历史天气。
    返回 Pandas DataFrame（时间索引 + 每个变量为一列）。
    """
    base_url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,   # 格式 YYYY-MM-DD
        "end_date": end_date,       # 格式 YYYY-MM-DD
        "hourly": hourly_vars,
        "timezone": timezone
    }

    resp = requests.get(base_url, params=params, timeout=30)
    try:
        resp.raise_for_status()
    except requests.HTTPError as e:
        # 显示服务器响应（若有 JSON 错误消息也打印）
        msg = f"HTTP error {resp.status_code}: {e}"
        try:
            err_json = resp.json()
            msg += f" | server message: {err_json}"
        except Exception:
            msg += f" | response text: {resp.text[:500]}"
        raise RuntimeError(msg)

    data = resp.json()

    # 检查返回结构并把 hourly 转为 DataFrame
    if "hourly" not in data:
        raise RuntimeError("API did not return 'hourly' data. Full response: " + str(data))

    hourly = data["hourly"]
    # hourly 应包含 time 数组和各变量数组
    df = pd.DataFrame(hourly)
    if "time" in df.columns:
        # 把 time 列设为索引并转换为 datetime
        df["time"] = pd.to_datetime(df["time"])
        df = df.set_index("time")
    return data, df

if __name__ == "__main__":
    # 巴塞罗那示例坐标
    lat, lon = 41.3851, 2.1734
    start_date = "2023-01-01"
    end_date   = "2023-01-05"
    try:
        raw_json, df = fetch_historical_weather(lat, lon, start_date, end_date)
        print("返回的顶级键：", list(raw_json.keys()))
        print("DataFrame 预览：")
        print(df.head())
        # 如需保存 CSV：
        df.to_csv("historical_weather.csv")
        print("保存为 historical_weather.csv")
    except Exception as err:
        print("请求或处理失败：", err)


返回的顶级键： ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
DataFrame 预览：
                     temperature_2m  precipitation
time                                              
2023-01-01 00:00:00             9.7            0.0
2023-01-01 01:00:00             8.6            0.0
2023-01-01 02:00:00             8.2            0.0
2023-01-01 03:00:00             7.6            0.0
2023-01-01 04:00:00             6.8            0.0
保存为 historical_weather.csv


In [6]:
# API endpoint and credentials
# Example for REE API (modify for your chosen API)

API_ENDPOINT = "https://apidatos.ree.es"  # Change if using different API
API_KEY = "YOUR_API_KEY_HERE"  # Add if your API requires authentication

# Define headers (modify based on your API)
headers = {
    'Accept': 'application/json',
    'Content-Type': 'application/json',
    'Host': 'apidatos.ree.es'
}

print("✓ API configuration set")

✓ API configuration set


---

## Part 3: Data Extraction Functions

Create reusable functions for fetching and processing data

In [ ]:
def fetch_data(endpoint, params=None):
    """
    Fetch data from API with error handling
    
    Args:
        endpoint (str): API endpoint URL
        params (dict): Query parameters
    
    Returns:
        dict: JSON response or None if failed
    """
    try:
        # send full URL
        full_url = API_ENDPOINT + endpoint

        # send request
        response = requests.get(full_url, headers=headers, params=params, timeout=10)

        # check status code
        if response.status_code == 200:
            print("✓ Data fetched successfully")
            return response.json()
        else:
            print(f"✗ Error: Status code {response.status_code}")
            print(f"Response: {response.text}")
            return None

    except requests.exceptions.Timeout:
        print("✗ Request timed out")
        return None
    except requests.exceptions.ConnectionError:
        print("✗ Connection error")
        return None
    except Exception as e:
        print(f"✗ Unexpected error: {e}")
        return None


# Test your function
# test_data = fetch_data("YOUR_ENDPOINT_HERE", params={"key": "value"})

In [ ]:
def extract_timeseries(data, include_prediction=False):
    """
    Extract time-series data from REE API response

    Args:
        data: API response data
        include_prediction: Whether to include prediction data

    Returns:
        tuple: (values, timestamps_as_datetime)
    """
    values = []
    timestamps = []

    try:
        # REE API data structure analysis
        if 'included' in data:
            for item in data['included']:
                if 'attributes' in item and 'values' in item['attributes']:
                    for value_item in item['attributes']['values']:
                        if 'value' in value_item and 'datetime' in value_item:
                            # choose contain or not data
                            if not include_prediction and value_item.get('value_type') == 'Previsión':
                                continue

                            values.append(value_item['value'])
                            # turn datetime to timestamp
                            timestamps.append(datetime.fromisoformat(
                                value_item['datetime'].replace('Z', '+00:00')
                            ))

        return values, timestamps
    except Exception as e:
        print(f"✗ Error extracting data: {e}")
        return [], []


print("✓ Helper functions defined")

---

## Part 4: Data Collection

Fetch the data you need for your three plots

In [ ]:
# Define date range for your analysis
start_date = "2023-11-01T00:00"  # Modify as needed
end_date = "2023-11-08T00:00"    # Modify as needed

# Set up parameters for your API request
params = {
    'start_date': start_date,
    'end_date': end_date,
    # Add other parameters as needed
}

print(f"Fetching data from {start_date} to {end_date}...")

# TODO: Fetch your data
# raw_data = fetch_data(YOUR_ENDPOINT, params)

# TODO: Process and extract time-series
# values, times = extract_timeseries(raw_data, 'value_key', 'time_key')

In [ ]:
# Explore your data structure
# Uncomment to see the structure of your API response

# print("Data structure:")
# pprint(raw_data, depth=2)

---

## Part 5: Plot #1 - Basic Time-Series Line Plot (20 points)

**Requirements:**
- Single variable over time
- Minimum 24 hours of data
- Proper title, axis labels, grid, and legend
- Include average line or trend indicator
- DateTime objects on x-axis

**Description:** [Describe what this plot shows]

In [ ]:
# TODO: Create your first plot

fig, ax = plt.subplots(figsize=(14, 6))

# Plot your data
# ax.plot(times, values, ...)

# Add average line
# avg = sum(values) / len(values)
# ax.axhline(y=avg, ...)

# Formatting
ax.set_title('YOUR TITLE HERE', fontsize=16, fontweight='bold')
ax.set_xlabel('Time', fontsize=13)
ax.set_ylabel('YOUR VARIABLE (UNITS)', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Print statistics
# print(f"Average: {avg:.2f}")
# print(f"Min: {min(values):.2f}")
# print(f"Max: {max(values):.2f}")

### Analysis for Plot #1

**Key Observations:**
- [What patterns do you see?]
- [Any interesting trends?]
- [What might explain the patterns?]

[Your analysis here]

---

## Part 6: Plot #2 - Comparative Time-Series (20 points)

**Requirements:**
- Compare at least 2 different time-series
- Clear differentiation (colors, line styles, markers)
- Legend explaining each series
- Brief interpretation of the comparison

**Description:** [Describe what this plot compares]

In [ ]:
# TODO: Fetch/prepare second dataset if needed
# data2 = fetch_data(...)
# values2, times2 = extract_timeseries(...)

In [ ]:
# TODO: Create your comparison plot

fig, ax = plt.subplots(figsize=(14, 6))

# Plot multiple series
# ax.plot(times1, values1, label='Series 1', ...)
# ax.plot(times2, values2, label='Series 2', ...)

ax.set_title('YOUR COMPARISON TITLE', fontsize=16, fontweight='bold')
ax.set_xlabel('Time', fontsize=13)
ax.set_ylabel('YOUR VARIABLE (UNITS)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Calculate comparison statistics
# difference = [abs(v1 - v2) for v1, v2 in zip(values1, values2)]
# print(f"Average difference: {sum(difference)/len(difference):.2f}")

### Analysis for Plot #2

**Comparison Insights:**
- [How do the two series relate?]
- [Are they correlated or inversely related?]
- [What causes the differences?]

[Your analysis here]

---

## Part 7: Plot #3 - Advanced Visualization (20 points)

**Requirements:**
- Aggregated or processed time-series data
- Advanced visualization type (stacked area, heatmap, box plot, dual-axis)
- Professional formatting
- Statistical analysis or key findings displayed

**Description:** [Describe your advanced visualization approach]

In [ ]:
# TODO: Prepare data for advanced visualization
# This might involve aggregation, grouping, or calculating derived metrics

# Example for stacked area:
# data_by_category = {...}  # Dictionary of categories and their values over time

# Example for heatmap:
# hourly_patterns = aggregate_by_hour_and_day(data)

# Example for box plot:
# daily_groups = group_by_day(data)

In [ ]:
# TODO: Create your advanced plot

fig, ax = plt.subplots(figsize=(15, 8))

# Choose ONE of these approaches:

# Option 1: Stacked Area Chart
# ax.stackplot(times, series1, series2, series3, labels=[...])

# Option 2: Heatmap
# import numpy as np
# im = ax.imshow(data_matrix, aspect='auto', cmap='viridis')
# plt.colorbar(im)

# Option 3: Box Plot by Time Period
# ax.boxplot([data_day1, data_day2, ...], labels=[...])

# Option 4: Dual-axis plot
# ax2 = ax.twinx()
# ax.plot(...)
# ax2.plot(...)

ax.set_title('YOUR ADVANCED PLOT TITLE', fontsize=16, fontweight='bold')
ax.set_xlabel('YOUR X LABEL', fontsize=13)
ax.set_ylabel('YOUR Y LABEL', fontsize=13)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Advanced statistical analysis
# TODO: Calculate relevant statistics for your visualization

# Examples:
# - Correlation coefficients
# - Percentage breakdown
# - Peak hour analysis
# - Variability measures

print("📊 Statistical Summary:")
# print(f"  Metric 1: {value}")
# print(f"  Metric 2: {value}")

### Analysis for Plot #3

**Advanced Insights:**
- [What patterns emerge from the aggregated view?]
- [What are the key statistical findings?]
- [What's the main story this visualization tells?]

[Your analysis here]

---

## Part 8: Overall Conclusions

**Summary of Findings:**

[Write a summary paragraph bringing together insights from all three plots]

**Challenges Encountered:**

[Describe any difficulties you faced and how you solved them]

**Potential Extensions:**

[What additional analysis could be interesting with this data?]

---

## Part 9: Code Quality Check

Before submitting, verify:

- [ ] All code cells run without errors
- [ ] All plots are properly labeled and formatted
- [ ] Functions include docstrings
- [ ] Error handling is implemented
- [ ] Comments explain complex logic
- [ ] README.md file is complete
- [ ] No hardcoded API keys (or marked clearly for replacement)
- [ ] All three plots meet the requirements

---

## Appendix: Helper Functions (Optional)

Add any additional helper functions you created:

In [ ]:
# Additional helper functions

def your_helper_function():
    """
    Description of what this function does
    """
    pass